In [17]:
from datetime import datetime
import meteostat as ms
import pandas as pd

start = datetime(2015, 1, 1)
end = datetime(2024, 12, 31)


def extract_yearly_climate(lat, lon, start, end, radius_meters=250000):

    point = ms.Point(lat, lon)

    stations = ms.stations.nearby(
        point,
        radius=radius_meters,
        limit=5
    )

    if stations.empty:
        return pd.DataFrame()

    ts = ms.daily(stations, start, end)

    df = ms.interpolate(ts, point).fetch()

    if df.empty:
        return pd.DataFrame()

    if "tavg" not in df.columns and "tmin" in df.columns and "tmax" in df.columns:
        df["tavg"] = (df["tmin"] + df["tmax"]) / 2

    df.index = pd.to_datetime(df.index)

    df["Year"] = df.index.year

    agg = {}

    if "tavg" in df.columns:
        agg["tavg"] = "mean"

    if "prcp" in df.columns:
        agg["prcp"] = "sum"

    if "wspd" in df.columns:
        agg["wspd"] = "mean"

    if "pres" in df.columns:
        agg["pres"] = "mean"

    if "tmin" in df.columns:
        agg["tmin"] = "min"  
        
    if "tmax" in df.columns:
        agg["tmax"] = "max"  

    yearly = (
        df.groupby("Year")
        .agg(agg)
        .reset_index()
    )

    return yearly

yearly = extract_yearly_climate(
    13.9299,
    75.5681,
    start,
    end
)

yearly

test.index = pd.to_datetime(test.index)

test["Year"] = test.index.year

yearly = (
    test.groupby("Year")
    .agg({
        "tavg": "mean",
        "tmin": "mean",
        "tmax": "mean",
        "prcp": "sum",
        "wspd": "mean",
        "pres": "mean"
    })
    .reset_index()
)

yearly


,Year,tavg,tmin,tmax,prcp,wspd,pres
0,2021,24.50087,17.666667,32.125,1779.7,8.916857,1010.823623
1,2022,24.812398,17.0,32.35,1452.4,9.229435,1010.377944
2,2023,25.869605,17.9,33.891667,751.4,11.694363,1011.567768
3,2024,25.963131,18.566667,34.033333,1359.0,10.783288,1010.704509


In [2]:
districts = pd.read_csv("../data/raw/outbreaks/district_master.csv")

districts.head()

,State,District,Latitude,Longitude,Forest_Type,Elevation_Category
0,Karnataka,Shivamogga,13.9299,75.5681,Tropical Moist Deciduous,Medium
1,Karnataka,Uttara Kannada,14.7937,74.6869,Tropical Evergreen,Medium
2,Karnataka,Chikkamagaluru,13.3161,75.7720,Tropical Evergreen,High
3,Karnataka,Kodagu,12.4244,75.7382,Tropical Evergreen,High
4,Kerala,Wayanad,11.6854,76.1320,Tropical Evergreen,High


In [3]:
records = []

In [20]:
for _, row in districts.iterrows():

    print(f"Processing {row['District']}...")

    yearly = extract_yearly_climate(
        row["Latitude"],
        row["Longitude"],
        start,
        end
    )

    if yearly.empty:
        continue

    yearly["State"] = row["State"]
    yearly["District"] = row["District"]

    records.append(yearly)

Processing Shivamogga...
Processing Uttara Kannada...
Processing Chikkamagaluru...
Processing Kodagu...
Processing Wayanad...
Processing Kannur...
Processing North Goa...
Processing South Goa...
Processing Sindhudurg...


In [21]:
climate_df = pd.concat(records, ignore_index=True)

climate_df.head()

,Year,tavg,prcp,wspd,pres,tmin,tmax,State,District
0,2021,24.506868,1779.7,8.926923,1010.807967,13.2,37.2,Karnataka,Shivamogga
1,2022,24.796006,1452.4,9.242424,1010.384298,12.8,37.0,Karnataka,Shivamogga
2,2023,25.862466,751.4,11.691781,1011.566575,13.0,39.3,Karnataka,Shivamogga
3,2024,25.99382,1359.0,10.76882,1010.710112,13.9,41.0,Karnataka,Shivamogga
4,2015,27.601681,3006.5,<NA>,<NA>,20.4,35.5,Karnataka,Uttara Kannada


In [23]:
cols = [
    "State",
    "District",
    "Year",
    "tavg",
    "tmin",
    "tmax",
    "prcp",
    "wspd",
    "pres"
]

existing_cols = [c for c in cols if c in climate_df.columns]

climate_df = climate_df[existing_cols]

In [24]:
print(climate_df.shape)

climate_df.head()

climate_df.tail()

(78, 9)


,State,District,Year,tavg,tmin,tmax,prcp,wspd,pres
73,Maharashtra,Sindhudurg,2020,27.721176,18.1,36.8,3299.6,<NA>,<NA>
74,Maharashtra,Sindhudurg,2021,26.933654,17.6,35.8,3211.7,10.497253,1009.279945
75,Maharashtra,Sindhudurg,2022,26.922329,16.0,37.3,2530.6,10.962534,1009.088705
76,Maharashtra,Sindhudurg,2023,27.667808,17.6,37.1,2176.7,15.435616,1010.348493
77,Maharashtra,Sindhudurg,2024,27.502809,16.7,37.0,3329.0,15.561345,1009.47451


In [28]:
climate_df.to_csv(
    "../data/processed/climate_western_ghats_2015_2024.csv",
    index=False
)

print("Climate dataset exported successfully!")

Climate dataset exported successfully!


In [25]:
print(climate_df.info())

print(climate_df.isnull().sum())

print(climate_df["District"].value_counts())

print(climate_df["Year"].unique())

<class 'pandas.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   State     78 non-null     str    
 1   District  78 non-null     str    
 2   Year      78 non-null     int32  
 3   tavg      78 non-null     Float64
 4   tmin      78 non-null     Float64
 5   tmax      78 non-null     Float64
 6   prcp      78 non-null     Float64
 7   wspd      54 non-null     Float64
 8   pres      44 non-null     Float64
dtypes: Float64(6), int32(1), str(2)
memory usage: 7.0 KB
None
State        0
District     0
Year         0
tavg         0
tmin         0
tmax         0
prcp         0
wspd        24
pres        34
dtype: int64
District
Uttara Kannada    10
Kodagu            10
Wayanad           10
Kannur            10
North Goa         10
South Goa         10
Sindhudurg        10
Shivamogga         4
Chikkamagaluru     4
Name: count, dtype: int64
[2021 2022 2023 2024 2015 2016 2017 2018 2019

In [26]:
print(climate_df.shape)

print(climate_df.head())

print(climate_df["Year"].unique())

print(climate_df["District"].unique())

(78, 9)
       State        District  Year       tavg  tmin  tmax    prcp       wspd  \
0  Karnataka      Shivamogga  2021  24.506868  13.2  37.2  1779.7   8.926923   
1  Karnataka      Shivamogga  2022  24.796006  12.8  37.0  1452.4   9.242424   
2  Karnataka      Shivamogga  2023  25.862466  13.0  39.3   751.4  11.691781   
3  Karnataka      Shivamogga  2024   25.99382  13.9  41.0  1359.0   10.76882   
4  Karnataka  Uttara Kannada  2015  27.601681  20.4  35.5  3006.5       <NA>   

          pres  
0  1010.807967  
1  1010.384298  
2  1011.566575  
3  1010.710112  
4         <NA>  
[2021 2022 2023 2024 2015 2016 2017 2018 2019 2020]
<ArrowStringArray>
[    'Shivamogga', 'Uttara Kannada', 'Chikkamagaluru',         'Kodagu',
        'Wayanad',         'Kannur',      'North Goa',      'South Goa',
     'Sindhudurg']
Length: 9, dtype: str


In [27]:
expected = 9 * 10

print("Expected:", expected)
print("Actual:", len(climate_df))
print("Missing:", expected - len(climate_df))

Expected: 90
Actual: 78
Missing: 12


In [29]:
climate_df.groupby("District")["Year"].count()

District
Chikkamagaluru     4
Kannur            10
Kodagu            10
North Goa         10
Shivamogga         4
Sindhudurg        10
South Goa         10
Uttara Kannada    10
Wayanad           10
Name: Year, dtype: int64

In [30]:
climate_df = climate_df.rename(columns={
    "tavg": "Temperature",
    "tmin": "Min_Temperature",
    "tmax": "Max_Temperature",
    "prcp": "Rainfall"
})

climate_df.head()

,State,District,Year,Temperature,Min_Temperature,Max_Temperature,Rainfall,wspd,pres
0,Karnataka,Shivamogga,2021,24.506868,13.2,37.2,1779.7,8.926923,1010.807967
1,Karnataka,Shivamogga,2022,24.796006,12.8,37.0,1452.4,9.242424,1010.384298
2,Karnataka,Shivamogga,2023,25.862466,13.0,39.3,751.4,11.691781,1011.566575
3,Karnataka,Shivamogga,2024,25.99382,13.9,41.0,1359.0,10.76882,1010.710112
4,Karnataka,Uttara Kannada,2015,27.601681,20.4,35.5,3006.5,<NA>,<NA>


In [31]:
climate_df.to_csv(
    "../data/processed/climate_western_ghats_2015_2024.csv",
    index=False
)

print("Climate dataset exported successfully!")

Climate dataset exported successfully!
